# Creating a rope bridge in c++ with Unreal Engine 5

## Introduction

In this document, I will detail the creation of a rope bridge. Additionally, the bridge will be procedurally generated.

## Description

To create the bridge, you will first need to create a C++ class which inherits from the AActor class (base class for objects in Unreal Engine). The bridge will consist of a SplinesComponent (a spline is a parametric curve) which will be generated procedurally using the functions of [chains](https://en.wikipedia.org/wiki/Catenary) (mathematical function which allows to model the shape of a rope). The spline will then be used to generate the deck planks. These may have random offsets to give a more natural appearance to the bridge.

## Steps

### 1 - Creation of the project

To begin, you must create a new Unreal Engine 5 project. To do this, simply launch the software and click on “New Project”. Then, you must choose the “Blank” template and select the c++ language. Other settings can be left as default.

<img src="Images/img1.png" alt="Create a new project" style="width: 300px;"/>

### 2 - Creating the bridge class

To create the bridge class, you must right-click in the project's "Source" folder and select "New C++ Class". Then, you must choose "Actor" as the parent of the class. The name of the class doesn't matter, but for this example I'll call it "RopesBridge".

<img src="Images/img2.png" alt="Create a new class" style="width: 200px;"/>
<br>
<img src="Images/img3.png" alt="Create a new class" style="width: 200px;"/>

We can immediately remove the BeginPlay and Tick functions from the class, because we are not going to use them.

### 3 - Ajout de la structure FCurveParams

When we open the class, we can see that 2 files have been created: RopesBridge.h and RopesBridge.cpp. The .h file contains the class declaration (the variables and function names) and the .cpp file contains the function implementation (the code that will be executed).

In the .h file, we will add an FCurveParams structure which will contain the parameters of the curve. This structure will contain the following variables:
- CatenaryAlpha, of type float, which represents the alpha parameter of the chain function. The default is 5.0f.
- CatenaryBeta, of type float, which represents the beta parameter of the chain function. The default is 17.5f.
- CatenaryBetaOffset, of type float, which represents the offset of the beta parameter of the chain function. The default is 0.0f.
- SplineDivision, of type int32, which represents the number of divisions of the spline. The default is 25.

This structure is placed just before the declaration of the RopesBridge class, in the .h file, and is declared as follows:

```c++
USTRUCT(BlueprintType)
struct FCurveParams
{
	GENERATED_BODY()

	UPROPERTY(BlueprintReadWrite, EditAnywhere, Category = "Curve properties", meta = (ToolTip = "The alpha parameter of the catenary equation. Defines the shape of the curve."))
	float CatenaryAlpha = 5.0f;

	UPROPERTY(BlueprintReadWrite, EditAnywhere, Category = "Curve properties", meta = (ToolTip = "The beta parameter of the catenary equation. Defines the portion of the curve used for the bridge."))
	float CatenaryBeta = 17.5f;

	UPROPERTY(BlueprintReadWrite, EditAnywhere, Category = "Curve properties", meta = (ToolTip = "The offset of the catenary equation. Defines the horizontal shift of the curve."))
	float CatenaryBetaOffset = 0.0f;

	UPROPERTY(BlueprintReadWrite, EditAnywhere, Category = "Curve properties")
	int SplineSubdivisions = 25;
};
```

### 4 - Added SplinesComponent, GenerateSpline function and associated variables

Still in the .h file, we will add the following components in the private section of the RopesBridge class:
```c++
UPROPERTY(VisibleAnywhere, BlueprintReadOnly, Category = "Bridge generation", meta = (AllowPrivateAccess = "true"))
USceneComponent* Root;

UPROPERTY(VisibleAnywhere, BlueprintReadOnly, Category = "Bridge generation", meta = (AllowPrivateAccess = "true"))
USplineComponent* Spline;
```

We can then add the following variables in the protected section of the RopesBridge class:
```c++
UPROPERTY(BlueprintReadWrite, EditAnywhere, Category = "Bridge properties")
float BridgeLength = 1000.0f;

UPROPERTY(BlueprintReadWrite, EditAnywhere, Category = "Bridge properties")
FCurveParams CurveParams;
```

Finally, we will add the GenerateSpline function in the protected section of the RopesBridge class, and the CalculateChainHeight and CalculateChainTangent functions in the private section of the RopesBridge class. The GenerateSpline function will generate the spline using the chain functions, and the CalculateChainHeight and CalculateChainTangent functions will calculate the height and tangent of the chain curve at a given point in space.

```c++
protected:
    UFUNCTION(BlueprintCallable, CallInEditor, Category = "Bridge generation")
    void GenerateSplines();

private:
    UFUNCTION(BlueprintCallable, Category = "Spline generation")
	float CalculateChainHeight(float X);

	UFUNCTION(BlueprintCallable, Category = "Spline generation")
	FVector CalculateChainTangent(FVector Previous, FVector Current, FVector Next);
```

At this stage, the .h file should look like this:

```c++
#pragma once

#include "CoreMinimal.h"
#include "GameFramework/Actor.h"
#include "RopeBridge.generated.h"

class USplineComponent;

USTRUCT(BlueprintType)
struct FCurveParams
{
	GENERATED_BODY()

	UPROPERTY(BlueprintReadWrite, EditAnywhere, Category = "Curve properties", meta = (ToolTip = "The alpha parameter of the catenary equation. Defines the shape of the curve."))
	float CatenaryAlpha = 5.0f;

	UPROPERTY(BlueprintReadWrite, EditAnywhere, Category = "Curve properties", meta = (ToolTip = "The beta parameter of the catenary equation. Defines the portion of the curve used for the bridge."))
	float CatenaryBeta = 17.5f;

	UPROPERTY(BlueprintReadWrite, EditAnywhere, Category = "Curve properties", meta = (ToolTip = "The offset of the catenary equation. Defines the horizontal shift of the curve."))
	float CatenaryBetaOffset = 0.0f;

	UPROPERTY(BlueprintReadWrite, EditAnywhere, Category = "Curve properties")
	int SplineSubdivisions = 25;
};

UCLASS()
class ROPESBRIDGE_API ARopeBridge : public AActor
{
	GENERATED_BODY()
	
	UPROPERTY(VisibleAnywhere, BlueprintReadOnly, Category = "Bridge generation", meta = (AllowPrivateAccess = "true"))
	USceneComponent* Root;

	UPROPERTY(VisibleAnywhere, BlueprintReadOnly, Category = "Bridge generation", meta = (AllowPrivateAccess = "true"))
	USplineComponent* Spline;
	
public:	
	ARopeBridge();

protected:
	UPROPERTY(BlueprintReadWrite, EditAnywhere, Category = "Bridge properties")
	float BridgeLength = 1000.0f;

	UPROPERTY(BlueprintReadWrite, EditAnywhere, Category = "Bridge properties")
	FCurveParams CurveParams;

	UFUNCTION(BlueprintCallable, CallInEditor, Category = "Bridge generation")
	void GenerateSplines();

private:
	UFUNCTION(BlueprintCallable, Category = "Spline generation")
	float CalculateChainHeight(float X);

	UFUNCTION(BlueprintCallable, Category = "Spline generation")
	FVector CalculateChainTangent(FVector Previous, FVector Current, FVector Next);
};
```

### 5 - Implementing the class constructor

The class constructor is a function that is called when the object is created. In the .cpp file, we will implement the constructor of the RopesBridge class. We will create the Root and Spline components, and we will attach them to the object.

```c++
ARopesBridge::ARopesBridge()
{
	PrimaryActorTick.bCanEverTick = false;

	Root = CreateDefaultSubobject<USceneComponent>(TEXT("Root"));
	RootComponent = Root;

	Spline = CreateDefaultSubobject<USplineComponent>(TEXT("Spline"));
	Spline->SetupAttachment(Root);
}
```

### 6 - Implementing the GenerateSpline function and related functions

In the .cpp file, we will implement the GenerateSpline, CalculateChainHeight and CalculateChainTangent functions. The GenerateSpline function will generate the spline using the chain functions, and the CalculateChainHeight and CalculateChainTangent functions will calculate the height and tangent of the chain curve at a given point in space.

```c++
ARopeBridge::ARopeBridge()
{
	PrimaryActorTick.bCanEverTick = false;

	Root = CreateDefaultSubobject<USceneComponent>(TEXT("Root"));
	RootComponent = Root;

	Spline = CreateDefaultSubobject<USplineComponent>(TEXT("Spline"));
	Spline->SetupAttachment(Root);
}

void ARopeBridge::GenerateSplines()
{
	Spline->ClearSplinePoints(false);
		
	float HeightOffset = CalculateChainHeight(0.0f);
	for (int i = 0; i < CurveParams.SplineSubdivisions; i++)
	{
		FVector Location = FVector(BridgeLength * i / (CurveParams.SplineSubdivisions - 1), 0.0f, CalculateChainHeight(i / (float)(CurveParams.SplineSubdivisions - 1)) - HeightOffset);
		Spline->AddSplinePoint(Location, ESplineCoordinateSpace::Local, false);
	}

	for (int i = 0; i < CurveParams.SplineSubdivisions; i++)
	{
		FVector Tangent = FVector(0.0f, 0.0f, 0.0f);

		if (i != 0 && i < CurveParams.SplineSubdivisions - 1)
		{
			FVector PreviousLocation = Spline->GetLocationAtSplinePoint(i - 1, ESplineCoordinateSpace::Local);
			FVector CurrentLocation = Spline->GetLocationAtSplinePoint(i, ESplineCoordinateSpace::Local);
			FVector NextLocation = Spline->GetLocationAtSplinePoint(i + 1, ESplineCoordinateSpace::Local);

			Tangent = CalculateChainTangent(PreviousLocation, CurrentLocation, NextLocation);
		}
		
		Spline->SetTangentAtSplinePoint(i, Tangent, ESplineCoordinateSpace::Local, false);
	}
}

float ARopeBridge::CalculateChainHeight(float X)
{
	float alpha = (-CurveParams.CatenaryBeta + CurveParams.CatenaryBetaOffset) + X * ((CurveParams.CatenaryBeta + CurveParams.CatenaryBetaOffset) - (-CurveParams.CatenaryBeta + CurveParams.CatenaryBetaOffset));
	return CurveParams.CatenaryAlpha * FMath::Cosh(alpha / CurveParams.CatenaryAlpha);
}

FVector ARopeBridge::CalculateChainTangent(FVector Previous, FVector Current, FVector Next)
{
	FVector PreviousToCurrent = Current - Previous;
	FVector CurrentToNext = Next - Current;

	return (PreviousToCurrent + CurrentToNext) / 2.0f;
}
```

The GenerateSpline function will first erase the spline points, then it will generate the x spline points using the chain functions. The CalculateChainHeight function will calculate the height of the chain curve at a given point, and the CalculateChainTangent function will calculate the tangent of the chain curve at this same point.

### 7 - Compiling and testing

To compile the project, simply click on the hammer icon in Rider. If all goes well, the project should compile without errors. We can then add a RopesBridge object into the scene by right-clicking in the scene window and selecting "RopesBridge", and adjusting the settings in the details panel, then clicking the "Generate Splines" button, we see the chain curve appear in the scene.

<video controls src="Videos/vid1.mp4" />

### 8 - Adding the bridge planks

To give more realism to the bridge, we will use several boards positioned along the spline, with small random offsets. To do this, we will add the FPlankParams structure in the .h file, which will contain the parameters of a plank.

```c++
USTRUCT(BlueprintType)
struct FPlankParams
{
	GENERATED_BODY()

	UPROPERTY(BlueprintReadWrite, EditAnywhere, Category = "Planks properties")
    UStaticMesh* PlankMesh;

	UPROPERTY(BlueprintReadWrite, EditAnywhere, Category = "Planks properties")
    FVector LocationOffset = FVector(0.0f, 0.0f, 0.0f);

	UPROPERTY(BlueprintReadWrite, EditAnywhere, Category = "Planks properties")
    FRotator RotationOffset = FRotator(0.0f, 0.0f, 0.0f);

	UPROPERTY(BlueprintReadWrite, EditAnywhere, Category = "Planks properties")
    FVector MinScale = FVector(0.9f, 0.9f, 0.9f);

	UPROPERTY(BlueprintReadWrite, EditAnywhere, Category = "Planks properties")
    FVector MaxScale = FVector(1.1f, 1.1f, 1.1f);

	UPROPERTY(BlueprintReadWrite, EditAnywhere, Category = "Planks properties")
    FRotator RandomRotation = FRotator(0.0f, 0.0f, 0.0f);

	UPROPERTY(BlueprintReadWrite, EditAnywhere, Category = "Planks properties")
    bool ApplyRandomToExtremities = false;
};
```

We can then create an array of FPlankParams in the protected section of the RopesBridge class, which will list the parameters for each type of plank used by the bridge. We also add a RandomSeed variable which will contain the seed of the random number generator (a seed is a number which allows you to generate a sequence of random numbers. The same seed will always give the same sequence of random numbers). Finally, we add a MissingPlankChance variable which will define the probability that a plank is missing.

```c++
protected:
    UPROPERTY(BlueprintReadWrite, EditAnywhere, Category = "Bridge properties")
    TArray<FPlanksParams> Planks;

	UPROPERTY(BlueprintReadWrite, EditAnywhere, Category = "Bridge properties")
    int RandomSeed = -1;

	UPROPERTY(BlueprintReadWrite, EditAnywhere, Category = "Bridge properties")
    float MissingPlankChance = 0.1f;
```

Then, in the pivate section of the class, we will add the PlanksComponents and PLanksIndexInSpline tables, which will contain the components of the boards and their index in the spline. We also add a FRandomStream type variable which will contain the random number generator, and the GeneratePlank, RandomInt and RandomFloat functions.

```c++
private:
    TArray<UStaticMeshComponent*> PlankComponents;
	TArray<int> PlanksIndexInSpline;

    FRandomStream RandomEngine;

	UFUNCTION(BlueprintCallable, Category = "Random engine")
	int RandomInt(int Min, int Max);

	UFUNCTION(BlueprintCallable, Category = "Random engine")
	float RandomFloat(float Min, float Max);

    UFUNCTION(BlueprintCallable, Category = "Planks generation")
	void GeneratePlank(FVector Location, FRotator Rotation, int Index);
```

Finally, we will add the GeneratePlanks function in the protected section of the RopesBridge class, which will generate the bridge planks using the parameters of each plank type.

```c++
protected:
    UFUNCTION(BlueprintCallable, CallInEditor, Category = "Bridge generation")
    void GeneratePlanks();
```

The .h file should look like this:

```c++
#pragma once

#include "CoreMinimal.h"
#include "GameFramework/Actor.h"
#include "RopeBridge.generated.h"

class USplineComponent;

USTRUCT(BlueprintType)
struct FCurveParams
{
	GENERATED_BODY()

	UPROPERTY(BlueprintReadWrite, EditAnywhere, Category = "Curve properties", meta = (ToolTip = "The alpha parameter of the catenary equation. Defines the shape of the curve."))
	float CatenaryAlpha = 5.0f;

	UPROPERTY(BlueprintReadWrite, EditAnywhere, Category = "Curve properties", meta = (ToolTip = "The beta parameter of the catenary equation. Defines the portion of the curve used for the bridge."))
	float CatenaryBeta = 17.5f;

	UPROPERTY(BlueprintReadWrite, EditAnywhere, Category = "Curve properties", meta = (ToolTip = "The offset of the catenary equation. Defines the horizontal shift of the curve."))
	float CatenaryBetaOffset = 0.0f;

	UPROPERTY(BlueprintReadWrite, EditAnywhere, Category = "Curve properties")
	int SplineSubdivisions = 25;
};

USTRUCT(BlueprintType)
struct FPlankParams
{
	GENERATED_BODY()

	UPROPERTY(BlueprintReadWrite, EditAnywhere, Category = "Planks properties")
	UStaticMesh* PlankMesh;

	UPROPERTY(BlueprintReadWrite, EditAnywhere, Category = "Planks properties")
	FVector LocationOffset = FVector(0.0f, 0.0f, 0.0f);

	UPROPERTY(BlueprintReadWrite, EditAnywhere, Category = "Planks properties")
	FRotator RotationOffset = FRotator(0.0f, 0.0f, 0.0f);

	UPROPERTY(BlueprintReadWrite, EditAnywhere, Category = "Planks properties")
	FVector MinScale = FVector(0.9f, 0.9f, 0.9f);

	UPROPERTY(BlueprintReadWrite, EditAnywhere, Category = "Planks properties")
	FVector MaxScale = FVector(1.1f, 1.1f, 1.1f);

	UPROPERTY(BlueprintReadWrite, EditAnywhere, Category = "Planks properties")
	FRotator RandomRotation = FRotator(0.0f, 0.0f, 0.0f);

	UPROPERTY(BlueprintReadWrite, EditAnywhere, Category = "Planks properties")
	bool ApplyRandomToExtremities = false;
};

UCLASS()
class ROPESBRIDGE_API ARopeBridge : public AActor
{
	GENERATED_BODY()
	
	UPROPERTY(VisibleAnywhere, BlueprintReadOnly, Category = "Bridge generation", meta = (AllowPrivateAccess = "true"))
	USceneComponent* Root;

	UPROPERTY(VisibleAnywhere, BlueprintReadOnly, Category = "Bridge generation", meta = (AllowPrivateAccess = "true"))
	USplineComponent* Spline;
	
public:	
	ARopeBridge();

protected:
	UPROPERTY(BlueprintReadWrite, EditAnywhere, Category = "Bridge properties")
	float BridgeLength = 1000.0f;

	UPROPERTY(BlueprintReadWrite, EditAnywhere, Category = "Bridge properties")
	FCurveParams CurveParams;

	UFUNCTION(BlueprintCallable, CallInEditor, Category = "Bridge generation")
	void GenerateSplines();

	UPROPERTY(BlueprintReadWrite, EditAnywhere, Category = "Bridge properties")
	TArray<FPlankParams> Planks;

	UPROPERTY(BlueprintReadWrite, EditAnywhere, Category = "Bridge properties")
	int RandomSeed = -1;

	UPROPERTY(BlueprintReadWrite, EditAnywhere, Category = "Bridge properties")
	float MissingPlankChance = 0.1f;

	UFUNCTION(BlueprintCallable, CallInEditor, Category = "Bridge generation")
	void GeneratePlanks();

private:
	UFUNCTION(BlueprintCallable, Category = "Spline generation")
	float CalculateChainHeight(float X);

	UFUNCTION(BlueprintCallable, Category = "Spline generation")
	FVector CalculateChainTangent(FVector Previous, FVector Current, FVector Next);

	TArray<UStaticMeshComponent*> PlankComponents;
	TArray<int> PlanksIndexInSpline;

	FRandomStream RandomEngine;

	UFUNCTION(BlueprintCallable, Category = "Random engine")
	int RandomInt(int Min, int Max);

	UFUNCTION(BlueprintCallable, Category = "Random engine")
	float RandomFloat(float Min, float Max);

	UFUNCTION(BlueprintCallable, Category = "Planks generation")
	void GeneratePlank(FVector Location, FRotator Rotation, int Index);
};
```

### 9 - Implementation of plank generation functions

In the .cpp file, we will start by implementing the RandomInt and RandomFloat functions, which will generate a random integer or floating number between two limits.

```c++
int ARopeBridge::RandomInt(int Min, int Max) { return RandomEngine.RandRange(Min, Max); }
float ARopeBridge::RandomFloat(float Min, float Max) { return RandomEngine.FRandRange(Min, Max); }
```

Next, we will implement the GeneratePlank function, which will generate a plank at a given position. We will choose a random type of board, then we will generate a position and a random rotation for the board. We will then create a StaticMeshComponent for the board, and we will attach it to the Root component.

```c++
void ARopeBridge::GeneratePlank(FVector Location, FRotator Rotation, int Index)
{
	FPlanksParams CurrentPlank = Planks[0];
	if (Planks.Num() > 1) { CurrentPlank = Planks[RandomInt(0, Planks.Num() - 1)]; }
	
	UStaticMeshComponent* PlankComponent = NewObject<UStaticMeshComponent>(Root);
	PlankComponent->SetStaticMesh(CurrentPlank.PlankMesh);
	PlankComponent->SetWorldLocation(Location + CurrentPlank.LocationOffset);
	PlankComponent->SetWorldRotation(Rotation);
	PlankComponent->AddLocalRotation(CurrentPlank.RotationOffset);
	PlankComponent->SetWorldScale3D(FVector(CurrentPlank.MaxScale));

	if (!(Index == 0 || Index == CurveParams.SplineSubdivisions - 1) || CurrentPlank.ApplyRandomToExtremities)
	{
		FRotator RandomRotation = FMath::Lerp(FRotator(0.0f, 0.0f, 0.0f), CurrentPlank.RandomRotation, RandomFloat(0.0f, 1.0f));
		if (RandomInt(0, 1)) { RandomRotation *= -1; }
		PlankComponent->AddLocalRotation(RandomRotation);

		FVector Scale = FMath::Lerp(FVector(CurrentPlank.MinScale), FVector(CurrentPlank.MaxScale), RandomFloat(0.0f, 1.0f));
		PlankComponent->SetWorldScale3D(Scale);
	}
	
	PlankComponent->AttachToComponent(Root, FAttachmentTransformRules::KeepRelativeTransform);
	PlankComponent->RegisterComponent();
	PlankComponents.Add(PlankComponent);
	PlanksIndexInSpline.Add(Index);

}
```

Finally, we will implement the GeneratePlanks function, which will generate the deck planks using the parameters of each type of plank. We will first delete the already generated boards if they exist, then we will generate the boards along the spline using the GeneratePlank and RandomInt functions.

```c++
void ARopeBridge::GeneratePlanks()
{
	for (UStaticMeshComponent* PlankComponent : PlankComponents) { PlankComponent->DestroyComponent(); }
	PlankComponents.Empty(); PlanksIndexInSpline.Empty();

	if (Planks.Num() < 1 || Planks[0].PlankMesh == nullptr) { UE_LOG(LogTemp, Error, TEXT("No plank mesh found or no planks defined")); return; }
	
	for (int i = 0; i < CurveParams.SplineSubdivisions; i++)
	{
		FVector Location = Spline->GetLocationAtSplinePoint(i, ESplineCoordinateSpace::Local);
		FVector Tangent = Spline->GetTangentAtSplinePoint(i, ESplineCoordinateSpace::Local);
		FRotator Rotation = Tangent.Rotation();

		if (RandomFloat(0.0f, 1.0f) > MissingPlankChance || i == 0 || i == CurveParams.SplineSubdivisions - 1) { GeneratePlank(Location, Rotation, i); }
	}
}
```

### 10 - Compiling and testing

We can now recompile the project in the same way as before. By playing with the settings in the RopesBridge object details panel, we can generate the bridge planks by clicking on the "Generate Planks" button.

<img src="Images/img6.png" alt="Generate planks" style="width: 300px;"/>